# Random Forest Vehicle Breakdown Prediction
## Predicting 30-Day Vehicle Breakdown Risk

**Objective**: Develop a Random Forest model to predict whether an active vehicle will experience corrective maintenance (breakdown) within the next 30 days.

**Team Context**: This Random Forest model will be compared with KNN and XGBoost.

**Business Context**: Early breakdown detection allows preventive maintenance scheduling, reducing emergency repair costs and vehicle downtime.

**Output**: The model returns a list of all unique `ASSET_CODE_encoded` registered entries, along with a Boolean categorical value (yes/no) indicating whether the vehicle will break down, and a probability score that quantifies the model's confidence in each prediction.

## 1. Metric Selection and Justification

For vehicle breakdown prediction, we select metrics based on business impact:

### Primary Metrics:

**1. AUC-ROC (Area Under Curve)**
- Measures model's ability to distinguish between breakdown vs no breakdown
- Values: 0.5 = random, 1.0 = perfect
- **Why important**: Overall model quality assessment

**2. Recall (Sensitivity)**
- Percentage of actual breakdowns correctly identified
- Formula: True Positives / (True Positives + False Negatives)
- **Why critical**: Missing a breakdown = emergency repair, safety risk, high cost

**3. Precision**
- Percentage of breakdown predictions that were correct
- Formula: True Positives / (True Positives + False Positives)
- **Why important**: Too many false alarms = wasted maintenance resources

**4. F1-Score**
- Harmonic mean of Precision and Recall
- Formula: 2 × (Precision × Recall) / (Precision + Recall)
- **Why useful**: Balances catching breakdowns vs avoiding false alarms

### Business Priority:
**Recall > F1-Score > Precision > AUC**

Missing a breakdown is worse than unnecessary maintenance.

# 2. Import Libraries

We begin by importing the key libraries needed for this analysis.

In [ ]:
# Libraries for data management

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
import warnings
warnings.filterwarnings('ignore')

# Machine learning libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, classification_report, confusion_matrix, precision_recall_curve
)

# For understanding what the model learned
import shap
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully")

## 3. Data Loading and Preparation

This function loads the maintenance dataset along with the vehicle information by opening two essential files:

1. The main spreadsheet containing all maintenance records.

2. A JSON file providing the label for each license plate: The ``load_data`` function returns a dictionary that maps each encoded vehicle asset code to its original license plate, enabling clear interpretation and accurate vehicle identification in the model’s output.

In [ ]:
def load_data():
    print("Loading data...")

    # Load the main maintenance records
    df = pd.read_excel('data/SERVICE_ORDER_BASE_clean.xlsx')
    print(f"Loaded {len(df):,} maintenance records")

    # Load the vehicle name mappings
    with open('data/code_name_mappings.json', 'r') as f:
        mappings = json.load(f)
    vehicle_names = mappings.get('ASSET_CODE_encoded_to_original', {})

    return df, vehicle_names

# Actually load our data now
df, vehicle_names = load_data()

## 4. Target Variable Creation

This section defines the core of our problem by creating the target variable: **predicting whether a vehicle will break down within the next 30 days**.
    
For every maintenance record, we look into the future and check:
- Did this vehicle have a breakdown (corrective maintenance) in the next 30 days?
- If yes, target = 1 (breakdown will happen)
- If no, target = 0 (no breakdown expected)
    
This creates our training examples for the machine learning model.
    

In [ ]:
def create_breakdown_target(df):

    print("Creating breakdown targets...")

    # First, clean up the date information as we need valid dates
    df_clean = df.dropna(subset=['SERVICE_ORDER_year', 'SERVICE_ORDER_month', 'SERVICE_ORDER_day']).copy()
    print(f"After removing missing dates: {len(df_clean):,} records")

    # Remove obviously invalid dates (like year 1900 or month 15)
    df_clean = df_clean[
        (df_clean['SERVICE_ORDER_year'] >= 2000) &  # No dates before year 2000
        (df_clean['SERVICE_ORDER_year'] <= 2030) &  # No dates after 2030
        (df_clean['SERVICE_ORDER_month'] >= 1) &    # Month must be 1-12
        (df_clean['SERVICE_ORDER_month'] <= 12) &
        (df_clean['SERVICE_ORDER_day'] >= 1) &      # Day must be 1-31
        (df_clean['SERVICE_ORDER_day'] <= 31)
    ]
    print(f"After removing invalid dates: {len(df_clean):,} records")

    # Convert the separate year, month, day columns into proper date objects as this makes it easier to calculate
    try:
        df_clean['service_date'] = pd.to_datetime(
            df_clean[['SERVICE_ORDER_year', 'SERVICE_ORDER_month', 'SERVICE_ORDER_day']],
            errors='coerce'
        )
    except:
        # If the above method fails, try a different approach
        df_clean['service_date'] = pd.to_datetime(
            df_clean['SERVICE_ORDER_year'].astype(str) + '-' +
            df_clean['SERVICE_ORDER_month'].astype(str).str.zfill(2) + '-' +
            df_clean['SERVICE_ORDER_day'].astype(str).str.zfill(2),
            errors='coerce'
        )

    # Remove any records where date conversion failed
    df_clean = df_clean.dropna(subset=['service_date'])
    # Sort by vehicle and date - this makes the next step much faster
    df_clean = df_clean.sort_values(['ASSET_CODE_encoded', 'service_date'])
    print(f"Final valid records: {len(df_clean):,}")

    # For each record, check if breakdown happens in next 30 days
    results = []

    # Process each vehicle separately
    for vehicle_id in df_clean['ASSET_CODE_encoded'].unique():
        vehicle_data = df_clean[df_clean['ASSET_CODE_encoded'] == vehicle_id]

        # For each maintenance record of this vehicle
        for idx, record in vehicle_data.iterrows():
            current_date = record['service_date']

            # Look 30 days into the future from this service date
            future_date = current_date + timedelta(days=30)
            future_records = vehicle_data[
                (vehicle_data['service_date'] > current_date) &
                (vehicle_data['service_date'] <= future_date)
            ]

            # Check if any future record is a breakdown
            # In our data: PREVENTIVE_CORRECTIVE MAINTENANCE = 0 means breakdown
            #              PREVENTIVE_CORRECTIVE MAINTENANCE = 1 means planned maintenance
            will_breakdown = (future_records['PREVENTIVE_CORRECTIVE MAINTENANCE'] == 0).any()

            # Save this example with its features and target
            results.append({
                'vehicle_id': vehicle_id,
                'service_date': current_date,
                'target': int(will_breakdown), 
                
                # Basic features about the vehicle and current service
                'model_type': record['MODEL_TYPE_CODE_encoded'],
                'tier': record['TIER'],
                'asset_status': record['ASSET STATUS'],
                'maintenance_type': record['PREVENTIVE_CORRECTIVE MAINTENANCE'],
                'cost': record['GRAND TOTAL'],
                'product_code': record['PRODUCT_CODE_encoded']
            })

    # Convert our list of examples into a pandas DataFrame
    target_df = pd.DataFrame(results)
    print(f"Created {len(target_df):,} examples")
    print(f"Breakdown rate: {target_df['target'].mean():.1%}")  # What % will actually break down

    return target_df

# Create our target variable - this is what we want to predict
df_with_target = create_breakdown_target(df)

## 5. Feature Engineering - Creating smart inputs for our model

To help the model predict breakdowns, we turn raw data into meaningful features based on vehicle history and patterns. By transforming this information into a meaningful ‘vehicle health profile,’ the model gains structured signals that improve its ability to predict vulnerable vehicles more accurately.
    
We create features like:
- How old is this vehicle?
- How many times has it broken down recently?
- What are the cost trends?
- How long since the last breakdown?


In [ ]:
def create_features(df):

    print("Creating features...")
    
    df_features = df.copy()
    df_features = df_features.sort_values(['vehicle_id', 'service_date'])
    
    # FEATURE 1: Vehicle Age
    # Older vehicles tend to break down more often
    first_service = df_features.groupby('vehicle_id')['service_date'].min()
    df_features['first_service_date'] = df_features['vehicle_id'].map(first_service)
    df_features['vehicle_age_days'] = (df_features['service_date'] - df_features['first_service_date']).dt.days
    df_features = df_features.drop('first_service_date', axis=1)
    
    # FEATURE 2: Days Since Last Service
    # If a vehicle hasn't been serviced in a while, it might be more likely to break down
    df_features['days_since_last'] = df_features.groupby('vehicle_id')['service_date'].diff().dt.days
    df_features['days_since_last'] = df_features['days_since_last'].fillna(999)  # 999 = first service ever
    
    results = []

    # Creating more complex features by looking at each vehicle's history to understand patterns from the past
    for vehicle_id in df_features['vehicle_id'].unique():
        vehicle_data = df_features[df_features['vehicle_id'] == vehicle_id].copy()
        
        # For each service record of this vehicle
        for idx, row in vehicle_data.iterrows():
            current_date = row['service_date']
            
            # Look at different time windows in the past
            past_90_days = current_date - timedelta(days=90)  # 3 months back
            past_30_days = current_date - timedelta(days=30)  # 1 month back
            
            # Get historical records for these time windows
            hist_90 = vehicle_data[
                (vehicle_data['service_date'] >= past_90_days) & 
                (vehicle_data['service_date'] < current_date)
            ]
            hist_30 = vehicle_data[
                (vehicle_data['service_date'] >= past_30_days) & 
                (vehicle_data['service_date'] < current_date)
            ]
            all_previous = vehicle_data[vehicle_data['service_date'] < current_date]
            
            # FEATURE 3 & 4 & 5: Breakdown History
            # How many breakdowns in recent periods? This is a strong predictor
            recent_breakdowns_30d = (hist_30['maintenance_type'] == 0).sum()  # Last 30 days
            recent_breakdowns_90d = (hist_90['maintenance_type'] == 0).sum()  # Last 90 days
            total_breakdowns = (all_previous['maintenance_type'] == 0).sum()  # All time
            
            # FEATURE 6: Service Frequency
            # How active has maintenance been recently?
            recent_services = len(hist_90)
            
            # FEATURE 7 & 8: Cost Analysis
            # Are costs increasing? High costs might indicate serious problems
            avg_cost_30d = hist_30['cost'].mean() if len(hist_30) > 0 else 0
            avg_cost_90d = hist_90['cost'].mean() if len(hist_90) > 0 else 0
            cost_trend = avg_cost_30d - avg_cost_90d if avg_cost_90d > 0 else 0
            
            # FEATURE 9: Time Since Last Breakdown
            # Vehicles that haven't broken down in a while might be due
            last_breakdown = all_previous[all_previous['maintenance_type'] == 0]
            if len(last_breakdown) > 0:
                days_since_breakdown = (current_date - last_breakdown['service_date'].max()).days
            else:
                days_since_breakdown = 999  # Never had a breakdown before
            
            # FEATURE 10: Service Intensity
            # How often does this vehicle need service relative to its age?
            vehicle_age_months = max(row['vehicle_age_days'] / 30, 1)  # Prevent division by zero
            service_intensity = len(all_previous) / vehicle_age_months
            
            # Combine the original data with our new features
            row_dict = row.to_dict()
            row_dict.update({
                'recent_breakdowns_30d': recent_breakdowns_30d,
                'recent_breakdowns_90d': recent_breakdowns_90d,
                'total_breakdowns': total_breakdowns,
                'recent_services': recent_services,
                'avg_cost_30d': avg_cost_30d,
                'cost_trend': cost_trend,
                'days_since_breakdown': days_since_breakdown,
                'service_intensity': service_intensity
            })
            results.append(row_dict)
    
    # Convert back to DataFrame and clean up any missing values
    df_final = pd.DataFrame(results)
    df_final = df_final.fillna(0)  # Replace any NaN values with 0
    
    print(f"Created features for {len(df_final):,} records")
    return df_final

# Apply feature engineering to our data
df_with_features = create_features(df_with_target)

# These are the features (inputs) we'll use to train our model. 
feature_columns = [
    # Basic vehicle information
    'model_type', 'tier', 'asset_status', 'maintenance_type',
    'cost', 'product_code',
    
    # Our engineered features (the smart ones we created)
    'vehicle_age_days', 'days_since_last', 'days_since_breakdown',
    'recent_breakdowns_30d', 'recent_breakdowns_90d', 'total_breakdowns', 
    'recent_services', 'avg_cost_30d', 'cost_trend', 'service_intensity'
]

print(f"Using {len(feature_columns)} features for modeling")

## 6. Data Preparation for Modeling

This is a crucial step where we prepare the data using time-based splitting to avoid data leakage.  

**The period of time considered**  
It's also important to note that we only use recent data (last 2 years) since:
1. vehicle technology changes over time;
2. Maintenance practices evolve; 
3. Recent patterns are more relevant than old.

**The time-based split**  
We can't randomly split the data because the model would see future data to predict past events. To avoid it, we use time-based splitting:
 - Train on earlier dates: 80% of data
 - Test on later dates: 20% of data  



In [ ]:

df_sorted = df_with_features.sort_values('service_date')
recent_cutoff = df_sorted['service_date'].max() - timedelta(days=730)  # 730 days = 2 years
df_recent = df_sorted[df_sorted['service_date'] >= recent_cutoff]

print(f"Using recent data from: {recent_cutoff.strftime('%Y-%m-%d')} onwards")
print(f"Recent data records: {len(df_recent):,}")

# TIME-BASED SPLIT

split_point = int(len(df_recent) * 0.8)  # 80% for training
train_data = df_recent.iloc[:split_point]    # Earlier dates
test_data = df_recent.iloc[split_point:]     # Later dates

print(f"Training data: {len(train_data):,} records")
print(f"Test data: {len(test_data):,} records")

# SEPARATE FEATURES AND TARGET (X,Y)

# Training set
X_train = train_data[feature_columns]  # Features for training
y_train = train_data['target']         # Targets for training

# Test set  
X_test = test_data[feature_columns]    # Features for testing
y_test = test_data['target']           # Targets for testing

# Check the breakdown rates - should be similar in train and test
print(f"Training breakdown rate: {y_train.mean():.1%}")
print(f"Test breakdown rate: {y_test.mean():.1%}")

# If these rates are very different, it might indicate a problem with our data split

## 7. Enhanced Random Forest Model
We implement an optimized Random Forest model with class weighting and threshold tuning. Below are the main procedures we followed:

01. **Model Architecture Settings**: The model uses 300 trees, unrestricted depth (no artificial limits), and minimal samples per split to capture patterns aggressively, while considering √18 ≈ 4 random feature subsets to prevent overfitting.
02. **Class Imbalance Handling**: Since our data has more "no breakdown" than "breakdown" cases, we assign a higher weight to the breakdown class. Bootstrap sampling ensures robustness, and class weights address the imbalance, making missed breakdowns six times more costly than false alarms.
03. **Training the model**: The model is trained on the training dataset to learn patterns such as vehicles with multiple recent breakdowns being more likely to fail again.
04. **Prediction probabilities**: After training, we extract prediction probabilities rather than just binary outcomes, allowing us to assess the model’s confidence for each case. The model can return a percentage that can be interpreted as: "there is a 73% confidence that this vehicle will break down."
05. **Optimal Threshold**: By default, the model predicts a breakdown if the probability > 50%, but we can adjust this threshold to catch more breakdowns (at the cost of more false alarms). To generate the final predictions and evaluate performance, we used multiple metrics:

    - **AUC** to measure overall model quality (0.5 = random, 1.0 = perfect);
    - **Precision** to assess how accurate predicted breakdowns are ("Of predicted breakdowns, how many were correct?");
    - **Recall** to evaluate how many actual breakdowns we captured ("Of actual breakdowns, how many did we catch?");
    - **F1-score** to balance precision and recall;
    - Overall **accuracy** to summarize correctness (percentage value).

The results provide a concise yet comprehensive view of the model’s predictive performance.

In [ ]:
print("Training the model")

# Create our Random Forest model with optimized settings
rf_model = RandomForestClassifier(
    n_estimators=300,           
    max_depth=None,             
    min_samples_split=2,        # Split a node if it has at least 2 samples
    min_samples_leaf=1,         # A leaf can have just 1 sample
    max_features='sqrt',        
    
    # Class Imbalance Handling
    class_weight={0: 1, 1: 6},  # Normal case = weight 1, Breakdown = weight 6
    
    # Technical Settings
    bootstrap=True,             # Each tree trains on a random sample of data
    random_state=42,           # Makes results reproducible
    n_jobs=-1                  # Use all CPU cores for faster training
)

# TRAINING the model on our training data
rf_model.fit(X_train, y_train)
print("Model training complete")

# Getting PREDICTION PROBABILITIES
y_prob = rf_model.predict_proba(X_test)[:, 1]  # Get probability of breakdown (class 1)

# THRESHOLD OPTIMIZATION
print("\nOptimizing classification threshold")

# Calculate precision and recall for all possible thresholds
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
# F1-score balances precision and recall
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)

# Find the threshold that gives the best F1-score
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]
print(f"Optimal threshold found: {optimal_threshold:.3f}")

# OPTIMAL THRESHOLD 
# If probability ≥ threshold → predict breakdown
y_pred = (y_prob >= optimal_threshold).astype(int)

# Calculate all our performance metrics
final_metrics = {
    'AUC': roc_auc_score(y_test, y_prob),          
    'Precision': precision_score(y_test, y_pred),   
    'Recall': recall_score(y_test, y_pred),         
    'F1-Score': f1_score(y_test, y_pred),          
    'Accuracy': accuracy_score(y_test, y_pred)    
}

print("\n" + "="*50)
print("FINAL RANDOM FOREST PERFORMANCE")
print("="*50)
for metric, value in final_metrics.items():
    print(f"{metric:12}: {value:.3f}")

## 8. Performance Analysis and Visualization

This part is a detailed analysis of the model performance that helps us understand exactly how well our model is. 

### The Confusion Matrix

This is a performance evaluation tool that compares the model’s predictions with the actual outcomes, organizing results into categories of correct and incorrect classifications. We applied it because our problem is categorical rather than numerical, and this metric provides a clear view of performance in classification tasks.

This is how we can interpret the data shown:
- TRUE POSITIVES (tp): We correctly predicted these breakdowns
- TRUE NEGATIVES (tn): We correctly said these wouldn't break down  
- FALSE POSITIVES (fp): We predicted breakdown but it didn't happen (false alarm)
- FALSE NEGATIVES (fn): We missed these breakdowns (quite sad)

From the matrix, we can derive important measures such as:

- PRECISION: the proportion of predicted breakdowns that were truly breakdowns, showing how reliable the model is when it raises an alert.
- RECALL: the proportion of actual breakdowns that were successfully identified, reflecting how effective the model is at not missing real failures.

In [ ]:

# Create the confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()  # Extract the four numbers

print("DETAILED PERFORMANCE ANALYSIS")
print("=" * 50)

print(f"\nConfusion Matrix:")
print(f"                 Predicted")
print(f"               No    Yes")
print(f"Actual No   {tn:5d} {fp:5d}")  # Top row: vehicles that didn't break down
print(f"     Yes   {fn:5d} {tp:5d}")   # Bottom row: vehicles that did break down

# Business Impact Analysis
total_actual_breakdowns = fn + tp
breakdown_detection_rate = tp / total_actual_breakdowns if total_actual_breakdowns > 0 else 0

print(f"\nBusiness Impact:")
print(f"Breakdowns caught: {tp}/{total_actual_breakdowns} ({breakdown_detection_rate:.1%})")
print(f"Missed breakdowns: {fn} (emergency repairs needed)")
print(f"False alarms: {fp} (unnecessary maintenance)")

# Compare to baseline performance
baseline_accuracy = max(np.mean(y_test), 1 - np.mean(y_test))
print(f"\nModel Quality Assessment:")
print(f"Random Forest accuracy: {final_metrics['Accuracy']:.1%}")
print(f"Baseline (always guess most common): {baseline_accuracy:.1%}")

Finally, we can plot the matrix to reveal patterns and insights hidden in the raw data.

In [ ]:
plt.figure(figsize=(12, 8))

# CHART 1: Performance Metrics Bar Chart
plt.subplot(2, 2, 1)
metrics_names = list(final_metrics.keys())
metrics_values = list(final_metrics.values())
bars = plt.bar(metrics_names, metrics_values, color=['blue', 'green', 'orange', 'red', 'purple'])
plt.title('Random Forest Performance Metrics', pad=20)
plt.ylabel('Score')
plt.ylim(0, 1.1)  # Extended upper limit to prevent text overlap
plt.xticks(rotation=45)

# Add the actual values on top of each bar
for bar, value in zip(bars, metrics_values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{value:.3f}', ha='center', va='bottom', fontsize=9)

# CHART 2: Confusion Matrix Heatmap
plt.subplot(2, 2, 2)
ax = plt.gca()
# Create a heatmap showing the confusion matrix
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')

# Add a color bar to show what the colors mean
plt.colorbar(im, ax=ax, shrink=0.8)

# Add the numbers inside each cell
for i in range(2):
    for j in range(2):
        text = ax.text(j, i, f'{cm[i, j]}', 
                      ha='center', va='center', color='black', fontsize=12, fontweight='bold')

# Label the axes so we know what we're looking at
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['No Breakdown', 'Breakdown'])
ax.set_yticklabels(['No Breakdown', 'Breakdown'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')

# CHART 3: Business Impact Visualization
plt.subplot(2, 2, 3)
business_metrics = ['Caught Breakdowns', 'Missed Breakdowns', 'False Alarms']
business_values = [tp, fn, fp]
colors = ['green', 'red', 'orange']  # Green=good, Red=bad, Orange=cost

# Calculate appropriate y-axis limit with padding
max_value = max(business_values)
y_limit = max_value * 1.15  # Add 15% padding at the top

bars = plt.bar(business_metrics, business_values, color=colors)
plt.title('Business Impact Analysis', pad=20)
plt.ylabel('Number of Cases')
plt.ylim(0, y_limit)  # Set y-limit with padding
plt.xticks(rotation=45)

# Show the exact numbers on each bar
for bar, value in zip(bars, business_values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (max_value * 0.02), 
            f'{value}', ha='center', va='bottom', fontsize=9)

# CHART 4: Feature Importance
plt.subplot(2, 2, 4)
# Create a dataframe of feature importance scores
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False).head(10)  # Top 10 most important

# Create a horizontal bar chart (easier to read feature names)
plt.barh(range(len(feature_importance)), feature_importance['importance'])
plt.yticks(range(len(feature_importance)), feature_importance['feature'])
plt.xlabel('Feature Importance')
plt.title('Top 10 Most Important Features')
plt.gca().invert_yaxis()  # Put highest importance at the top

# Make the layout neat and show the charts
plt.tight_layout()
plt.show()

# Interpretation help:
print("\nChart Interpretation:")
print("- Performance Metrics: Higher bars = better (except we want all metrics balanced)")
print("- Confusion Matrix: Darker blue = more cases, we want high numbers in top-left and bottom-right")
print("- Business Impact: Green=success, Red=problems, Orange=costs")
print("- Feature Importance: Shows which vehicle characteristics matter most for predictions")

## 9. Model Explainability with SHAP

We use SHAP (Shapley Additive Explanations) to understand which features drive Random Forest predictions and why the model makes each prediction. To do this, we create a SHAP explainer specifically designed for tree-based models to analyze the internal decision-making process of our 300 trees.

For efficiency, we analyze a sample of 1,000 test examples, as processing all 17,000+ examples would be time-consuming.

In [ ]:
print("Creating SHAP explainer for the model")

explainer = shap.TreeExplainer(rf_model)

# Analyzes for our test data
X_explain = X_test.head(1000)
shap_values = explainer.shap_values(X_explain)

# Handle different SHAP output formats
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # We want explanations for the positive class (breakdowns)

print("SHAP analysis complete")

### Analyzing Feature Importance: Two Different Perspectives


First, we adjust the data format to ensure consistency regardless of how SHAP returns it. Next, the SHAP feature importance highlights the average impact of each feature on individual predictions and, finally, we see which features the model splits on most frequently across all trees.

In [ ]:
# Handle SHAP values dimension correctly
if isinstance(shap_values, list) and len(shap_values) == 2:
    # Binary classification returns list with values for each class
    shap_vals_for_positive_class = shap_values[1]
elif len(shap_values.shape) == 3:
    # If 3D array, take positive class (index 1)
    shap_vals_for_positive_class = shap_values[:, :, 1]
else:
    # Already correct format
    shap_vals_for_positive_class = shap_values

# SHAP Feature Importance
shap_feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'shap_importance': np.abs(shap_vals_for_positive_class).mean(0)
}).sort_values('shap_importance', ascending=False)

print("\nSHAP Feature Importance (Random Forest):")
print("=" * 50)
print("This shows how much each feature affects individual predictions on average:")
for idx, row in shap_feature_importance.head(12).iterrows():
    print(f"{row['feature']:25} {row['shap_importance']:.4f}")

# Standard Random Forest Feature Importance  
rf_feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'rf_importance': rf_model.feature_importances_
}).sort_values('rf_importance', ascending=False)

print("\nStandard Random Forest Feature Importance:")
print("=" * 50)
print("This shows which features are used most often for splitting in the decision trees:")
for idx, row in rf_feature_importance.head(12).iterrows():
    print(f"{row['feature']:25} {row['rf_importance']:.4f}")

print("\nUnderstanding the Difference:")
print("- SHAP Importance: How much each feature CHANGES individual predictions")
print("- RF Importance: How often each feature is USED to split data in trees")
print("- Both are useful! SHAP is better for explaining individual cases")
print("- RF Importance is better for understanding overall model behavior")

print("\nKey Insights:")
print("The most important features for predicting breakdowns are:")
top_3_features = rf_feature_importance.head(3)['feature'].tolist()
for i, feature in enumerate(top_3_features, 1):
    print(f"{i}. {feature.replace('_', ' ').title()}")
print("\nThis makes business sense - vehicle age, service patterns, and breakdown history")
print("are strong indicators of future breakdown risk")

## 10. Final Results Summary

This section is key to gain a deeper understanding of our model’s performance.

In [ ]:
print("\n" + "="*60)
print("FINAL RANDOM FOREST MODEL SUMMARY")
print("="*60)

print(f"\nModel Configuration:")
print(f"- Enhanced Random Forest with strong class weighting")
print(f"- n_estimators: 300, class_weight: {{0: 1, 1: 6}}")
print(f"- Optimal threshold: {optimal_threshold:.3f} (lower than 50% to catch more breakdowns)")
print(f"- Feature count: {len(feature_columns)} engineered features")

print(f"\nFinal Performance Metrics:")
for metric, value in final_metrics.items():
    # Add interpretation for each metric
    if metric == 'AUC':
        interpretation = "Overall model quality (0.5=random, 1.0=perfect)"
    elif metric == 'Precision':
        interpretation = "Of predicted breakdowns, how many were actually correct"
    elif metric == 'Recall':
        interpretation = "Of actual breakdowns, how many did we catch"
    elif metric == 'F1-Score':
        interpretation = "Balance between precision and recall"
    else:
        interpretation = "Overall accuracy"
    
    print(f"  {metric:<12}: {value:.3f} - {interpretation}")

print(f"\nBusiness Impact:")
print(f"- Breakdown detection rate: {breakdown_detection_rate:.1%}")
print(f"- Missed breakdowns: {fn} (these need emergency repairs)")
print(f"- False alarms: {fp:,} (unnecessary maintenance scheduled)")

# Create feature importance DataFrame for the summary
rf_feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'rf_importance': rf_model.feature_importances_
}).sort_values('rf_importance', ascending=False)

print(f"\nTop 5 Most Important Features:")
print("(These are the main factors the model uses to predict breakdowns)")
for idx, row in rf_feature_importance.head(5).iterrows():
    feature_name = row['feature'].replace('_', ' ').title()
    print(f"  {feature_name:25} {row['rf_importance']:.3f}")

## 10. False Positive Analysis

This investigation examines why the model generates false positives and whether they represent actual prediction errors or successful preventive interventions. We analyze whether vehicles predicted to break down received preventive maintenance, which would indicate the model successfully identified at-risk vehicles that were then proactively serviced.

In [ ]:

print("\n" + "="*60)
print("FALSE POSITIVE ANALYSIS")
print("="*60)

# Get all the cases where we predicted breakdown but it didn't happen in 30 days
fp_indices = np.where((y_pred == 1) & (y_test == 0))[0]
fp_data = test_data.iloc[fp_indices]

print(f"\nTotal False Positives: {len(fp_data):,}")

# INVESTIGATION: Preventive Maintenance
print(f"\nAnalyzing preventive interventions...")
preventive_interventions = []
for idx, row in fp_data.iterrows():
    current_date = row['service_date'] 
    vehicle_id = row['vehicle_id']
    
    # Look for any preventive maintenance in next 30 days
    vehicle_data = df_with_features[df_with_features['vehicle_id'] == vehicle_id]
    future_preventive = vehicle_data[
        (vehicle_data['service_date'] > current_date) &
        (vehicle_data['service_date'] <= current_date + timedelta(days=30)) &
        (vehicle_data['maintenance_type'] == 1)  # 1 = preventive maintenance
    ]
    
    if len(future_preventive) > 0:
        preventive_interventions.append(True)
    else:
        preventive_interventions.append(False)

preventive_count = sum(preventive_interventions)
preventive_rate = preventive_count / len(fp_data) * 100 if len(fp_data) > 0 else 0
remaining_fp = len(fp_data) - preventive_count
remaining_fp_rate = remaining_fp / len(fp_data) * 100 if len(fp_data) > 0 else 0

print(f"\nFALSE POSITIVE BREAKDOWN:")
print(f"- Received preventive maintenance: {preventive_count:,} ({preventive_rate:.1f}%)")
print(f"- No preventive maintenance: {remaining_fp:,} ({remaining_fp_rate:.1f}%)")

print(f"\nKEY INSIGHT:")
print(f"Of the {len(fp_data):,} 'false positives', {preventive_count:,} ({preventive_rate:.1f}%) received")
print(f"preventive maintenance - meaning the model successfully identified at-risk")
print(f"vehicles that were then proactively serviced to prevent breakdowns.")

# Calculate effective precision considering preventive maintenance as success
tp = ((y_pred == 1) & (y_test == 1)).sum()
effective_tp = tp + preventive_count
total_positive_predictions = (y_pred == 1).sum()
effective_precision = effective_tp / total_positive_predictions if total_positive_predictions > 0 else 0

print(f"\nADJUSTED PERFORMANCE:")
print(f"- Standard Precision: {final_metrics['Precision']:.1%}")
print(f"- Effective Precision (counting preventive maintenance as success): {effective_precision:.1%}")
print(f"- This represents a more accurate view of the model's business value")

print(f"\nCONCLUSION:")
print(f"The model's true performance is better than raw metrics suggest, as many")
print(f"'false positives' led to successful preventive interventions that likely")
print(f"prevented the predicted breakdowns from occurring.")